# Week 6: Multi-Table SQL Queries, Data Cleaning & Transformation
### CPSC 5071 Project, Week 6
**Dataset:** Kaggle Earthquake Data Source (2000–2025)

This Jupyter Notebook combines Parts B and C of the assignment:

| Part | Description                                             |
|------|---------------------------------------------------------|
| **A** | SQL JOIN queries (see `week6_multi_table_queries.sql`)  |
| **B** | Load one SQL query result into pandas and inspect       |
| **C** | Data cleaning and transformation (main focus)           |
| **D** | Reflection write-up (separately in a `readme.txt` file) |


---
## Part B: Load SQL Results into pandas

### Step 3 — Load One Query Result

We choose Query 2 from `week6_multi_table_queries.sql` as our main dataset.
This query JOINs geographic locations, seismic events, magnitude measurements, 
and magnitude types — giving us the richest view of the data for cleaning 
and analysis.


In [5]:
import sqlite3
import pandas as pd
import numpy as np

# ── Connect to the SQLite database ──
## Notes:
# - We assume that the earthquakes.db file is in the same directory as this notebook.
conn = sqlite3.connect("earthquake.db")

# ── We'll use Query 2: Returns Locations + Magnitudes + Magnitude Types ──
# Uses INNER JOIN (GEOGRAPHIC_LOCATION ↔ SEISMIC_EVENT) and
# LEFT JOINs to MAGNITUDE_MEASUREMENT and MAGNITUDE_TYPE to
# preserve rows even when measurement data is missing.
# We will read this from the file that was created in Part A of the project.

with open("week6_multi_table_queries.sql", "r") as f:
    sql_text = f.read()

# Extract Query 2 (the second query block in the file)
queries = sql_text.split(";")
query = queries[1].strip() + ";"

# We use pd.read_sql_query() rather than the generic pd.read_sql() wrapper
# because we are executing a specific SQL query, not reading an entire table.
# Both produce identical results, but read_sql_query() is the more explicit
# API for this use case.
df = pd.read_sql_query(query, conn)
conn.close()

print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns from the database.")


Loaded 106,077 rows × 13 columns from the database.


#### `df.head()` — We will list the first five rows here

In [6]:
df.head()

,location_id,latitude,longitude,depth_km,place_description,event_id,event_timestamp,last_updated,magnitude_value,measurement_error,num_stations_used,type_code,type_name
0,46465,57.5699,-149.0911,27.8,"190 km E of Chiniak, Alaska",ak000126digo,2000-01-23 08:42:28.405000+00:00,2022-04-29T18:29:54.135Z,5.5,NaN,NaN,mw,None
1,46438,65.0087,-154.2390,10.0,northern Alaska,ak0001kedehd,2000-02-03 10:24:57.773000+00:00,2022-04-29T18:30:52.396Z,5.6,NaN,NaN,mw,None
2,46359,60.2025,-145.9216,18.1,"39 km SSW of Cordova, Alaska",ak0002nyhqq3,2000-02-27 02:22:14.511000+00:00,2022-04-29T18:33:57.113Z,5.0,NaN,NaN,mw,None
3,46343,60.1896,-145.9005,10.0,"40 km S of Cordova, Alaska",ak0002t9i8pq,2000-03-01 23:05:14.227000+00:00,2022-04-29T19:23:12.999Z,5.4,NaN,NaN,mw,None
4,46328,57.3410,-154.2517,39.5,"27 km SW of Larsen Bay, Alaska",ak00034p02su,2000-03-08 14:20:57.427000+00:00,2022-04-29T18:34:34.872Z,5.4,NaN,NaN,mw,None


#### `df.info()` — We will gather column types, non-null counts, and memory usage here

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106077 entries, 0 to 106076
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   location_id        106077 non-null  int64  
 1   latitude           106077 non-null  float64
 2   longitude          106077 non-null  float64
 3   depth_km           105484 non-null  float64
 4   place_description  105855 non-null  object 
 5   event_id           106077 non-null  object 
 6   event_timestamp    106077 non-null  object 
 7   last_updated       106077 non-null  object 
 8   magnitude_value    106077 non-null  float64
 9   measurement_error  36554 non-null   float64
 10  num_stations_used  42353 non-null   float64
 11  type_code          106077 non-null  object 
 12  type_name          0 non-null       object 
dtypes: float64(6), int64(1), object(6)
memory usage: 10.5+ MB


#### `df.describe()` — List the summary statistics for numeric columns

In [8]:
df.describe()

,location_id,latitude,longitude,depth_km,magnitude_value,measurement_error,num_stations_used
count,106077.000000,106077.000000,106077.000000,105484.000000,106077.000000,36554.000000,42353.000000
mean,53021.742998,3.794865,40.265552,61.576360,5.452841,0.171523,54.878497
std,30608.277356,30.328921,121.991382,107.751084,0.485270,0.152104,82.993485
min,1.000000,-77.080000,-179.997000,-4.000000,5.000000,0.000000,0.000000
25%,26515.000000,-17.741000,-72.395000,12.000000,5.100000,0.060000,12.000000
50%,53027.000000,-0.569000,99.069000,33.000000,5.300000,0.100000,27.000000
75%,79532.000000,30.160000,142.820000,50.300000,5.700000,0.210000,61.000000
max,105951.000000,87.386000,180.000000,700.000000,9.500000,1.840000,1027.000000


#### `df.describe(include='all')` — Full summary including categorical columns

For this part of the project, we will also include a full summary of information.
This is optional based on our instruction, but simple enough to include.
This version of the call expands the statistical summary to string/object columns so we can see
unique counts, top values, and frequencies alongside the numeric distributions.


In [9]:
df.describe(include='all')

,location_id,latitude,longitude,depth_km,place_description,event_id,event_timestamp,last_updated,magnitude_value,measurement_error,num_stations_used,type_code,type_name
count,106077.000000,106077.000000,106077.000000,105484.000000,105855,106077,106077,106077,106077.000000,36554.000000,42353.000000,106077,0
unique,NaN,NaN,NaN,NaN,64262,106077,106076,101803,NaN,NaN,NaN,28,0
top,NaN,NaN,NaN,NaN,South Sandwich Islands region,ak000126digo,2021-02-27 18:59:25.296000+00:00,2018-06-04T20:43:44.000Z,NaN,NaN,NaN,mb,NaN
freq,NaN,NaN,NaN,NaN,2416,1,2,265,NaN,NaN,NaN,41766,NaN
mean,53021.742998,3.794865,40.265552,61.576360,NaN,NaN,NaN,NaN,5.452841,0.171523,54.878497,NaN,NaN
std,30608.277356,30.328921,121.991382,107.751084,NaN,NaN,NaN,NaN,0.485270,0.152104,82.993485,NaN,NaN
min,1.000000,-77.080000,-179.997000,-4.000000,NaN,NaN,NaN,NaN,5.000000,0.000000,0.000000,NaN,NaN
25%,26515.000000,-17.741000,-72.395000,12.000000,NaN,NaN,NaN,NaN,5.100000,0.060000,12.000000,NaN,NaN
50%,53027.000000,-0.569000,99.069000,33.000000,NaN,NaN,NaN,NaN,5.300000,0.100000,27.000000,NaN,NaN
75%,79532.000000,30.160000,142.820000,50.300000,NaN,NaN,NaN,NaN,5.700000,0.210000,61.000000,NaN,NaN


---
## Part C: Data Cleaning and Transformation (Main Focus)

Our raw dataset comes from multiple JOINed tables and contains several data quality
issues typical of real data source. The cleaning strategy below addresses
missing values, inconsistent formatting, incorrect data types, and derives new
analytical features.


### Step 4 — Handle Missing Data
#### `df.isnull().sum()` — Seeing which entities have missing data

In [10]:
df.isnull().sum()

location_id               0
latitude                  0
longitude                 0
depth_km                593
place_description       222
event_id                  0
event_timestamp           0
last_updated              0
magnitude_value           0
measurement_error     69523
num_stations_used     63724
type_code                 0
type_name            106077
dtype: int64

In [11]:
## Here we chose to drop the column of 'type_name' due to the high instance
# of missing values, which makes it unusable for meaningful analysis. Since
# there was insufficient data in the column, imputing values would not be appropriate.

df = df.drop(columns=['type_name'])

## Then we take all the columns with missing values that are numeric and 
## fill them with the median
## We fill missing numeric values with the median rather than the mean
## because seismic data distributions are right-skewed with significant
## outliers. The median is robust to these outliers and better
## represents the typical value. Dropping rows was not
## possible here. Columns like measurement_error and num_stations_used are
## 60–65% null, primarily in older events, and dropping them would eliminate
## the majority of the dataset and bias any time based analysis toward
## recent data only.

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
df.isnull().sum()


location_id            0
latitude               0
longitude              0
depth_km               0
place_description    222
event_id               0
event_timestamp        0
last_updated           0
magnitude_value        0
measurement_error      0
num_stations_used      0
type_code              0
dtype: int64

In [12]:
## Finally, we fill any null values in a categorical column with "Unknown"
## For categorical columns like place_description, we fill nulls with
## "Unknown" rather than dropping rows because only a small fraction
## (approximately 0.2%) are missing. A placeholder preserves these rows for numeric
## analysis while clearly signaling incomplete data in any location-based
## grouping or filtering.

categorical_cols = [col for col in df.columns if col not in numeric_cols]

for col in categorical_cols:
    if np.issubdtype(df[col].dtype, np.datetime64):
        continue
    df[col] = df[col].replace({"nan": np.nan, "NaT": np.nan})
    if df[col].isna().any():
        df[col] = df[col].fillna("Unknown")

df.isnull().sum()

location_id          0
latitude             0
longitude            0
depth_km             0
place_description    0
event_id             0
event_timestamp      0
last_updated         0
magnitude_value      0
measurement_error    0
num_stations_used    0
type_code            0
dtype: int64

### Step 5 — Clean and Standardized Columns

In [13]:
## Now we standardize and clean the rest of the columns by striping white space and converting the datetime 
## columns to date time format. 

for col in categorical_cols:
    df[col] = df[col].str.strip()

df['event_timestamp'] = pd.to_datetime(df['event_timestamp'], errors='coerce')
df['last_updated'] = pd.to_datetime(df['last_updated'], errors='coerce')

## Standardize capitalization of type_code:
## The raw magnitude type codes mix cases inconsistently. For example,
## 'Md' vs 'md', 'Ml' vs 'ml', and 'Mfa' vs 'mfa'. The seismological
## convention uses lowercase. Without this step, grouping or filtering
## by type_code would treat 'Md' and 'md' as separate categories.
df['type_code'] = df['type_code'].str.lower()


### Step 6 — Transform Data

In [14]:
## To transform this dataframe, we extracted the year and location of each earthquake 
## then grouped by location, year, type of earthquake, and how deep it was to better understand 
## the relationship between location, year, types, and depths.

df['event_year'] = df['event_timestamp'].dt.year

def location(text):
    if pd.isna(text):
        return "Unknown"
    text = str(text)
    if " of " in text:
        text = text.split(" of ", 1)[1]
    text = text.replace("Earthquake", "")
    text = text.strip()
    return text

df['location'] = df['place_description'].apply(location)

earthquakes_by_type_location_year = (
    df.groupby(['location', 'event_year', 'type_code'])
      .size()
      .reset_index(name='earthquake_count'))

## Display the grouped summary to show the relationship between
## location, year, and magnitude type.
print(f"Grouped summary: {earthquakes_by_type_location_year.shape[0]:,} unique combinations")
earthquakes_by_type_location_year.head(10)

df['depth_category'] = pd.cut(
    df['depth_km'],
    bins=[-1, 70, 300, df['depth_km'].max()],
    labels=['Shallow', 'Intermediate', 'Deep'])

df[['event_year', 'location', 'depth_category']].head()
## df.sort_values(by='event_year')

Grouped summary: 47,322 unique combinations


,event_year,location,depth_category
0,2000.0,"Chiniak, Alaska",Shallow
1,2000.0,northern Alaska,Shallow
2,2000.0,"Cordova, Alaska",Shallow
3,2000.0,"Cordova, Alaska",Shallow
4,2000.0,"Larsen Bay, Alaska",Shallow


In [15]:
# Export to CSV 
df.to_csv("cleaned_data.csv", index=False)

print("Data cleaning complete. 'cleaned_data.csv' has been generated.")
df.head()

Data cleaning complete. 'cleaned_data.csv' has been generated.


,location_id,latitude,longitude,depth_km,place_description,event_id,event_timestamp,last_updated,magnitude_value,measurement_error,num_stations_used,type_code,event_year,location,depth_category
0,46465,57.5699,-149.0911,27.8,"190 km E of Chiniak, Alaska",ak000126digo,2000-01-23 08:42:28.405000+00:00,2022-04-29 18:29:54.135000+00:00,5.5,0.1,27.0,mw,2000.0,"Chiniak, Alaska",Shallow
1,46438,65.0087,-154.2390,10.0,northern Alaska,ak0001kedehd,2000-02-03 10:24:57.773000+00:00,2022-04-29 18:30:52.396000+00:00,5.6,0.1,27.0,mw,2000.0,northern Alaska,Shallow
2,46359,60.2025,-145.9216,18.1,"39 km SSW of Cordova, Alaska",ak0002nyhqq3,2000-02-27 02:22:14.511000+00:00,2022-04-29 18:33:57.113000+00:00,5.0,0.1,27.0,mw,2000.0,"Cordova, Alaska",Shallow
3,46343,60.1896,-145.9005,10.0,"40 km S of Cordova, Alaska",ak0002t9i8pq,2000-03-01 23:05:14.227000+00:00,2022-04-29 19:23:12.999000+00:00,5.4,0.1,27.0,mw,2000.0,"Cordova, Alaska",Shallow
4,46328,57.3410,-154.2517,39.5,"27 km SW of Larsen Bay, Alaska",ak00034p02su,2000-03-08 14:20:57.427000+00:00,2022-04-29 18:34:34.872000+00:00,5.4,0.1,27.0,mw,2000.0,"Larsen Bay, Alaska",Shallow
